# Fink/LSST — Dipole Concentration per diaObject

This notebook investigates whether **dipole alerts are uniformly distributed across diaObjects**,
or whether certain objects concentrate the majority of dipoles.

## Scientific goals

1. **Dipole count histogram per object**: stacked bar histogram of the number of dipole alerts per
   `diaObjectId`, colour-coded by filter band (u/g/r/i/z/y), to reveal whether a small
   number of objects dominate the dipole budget.

2. **Selection of high-dipole objects**: identify diaObjects that simultaneously have
   - a large total number of visits across all bands (`n_visits >= MIN_VISITS_TOTAL`), AND
   - a large number of dipole detections (`n_dipoles >= MIN_DIPOLES`).
   The `MIN_VISITS_TOTAL` threshold (50–100) ensures that objects with only 2 visits
   and 2 dipoles are excluded as statistically uninformative.

3. **Light curve inspection of selected objects**: for each selected diaObject plot the full
   multi-band light curve (all visits, colour-coded by band) with dipole visits highlighted by
   grey marker outlines.  The x-axis shows MJD with a secondary date axis (YYYY-MM-DD).  A
   secondary y-axis shows the nightly dipole count histogram (any band).

## Data source

Parquet files produced by notebook `01c_fink_dipoles_per_ddf.ipynb` stored in
`data_DIPOLES_01c/`.  **No Fink API calls are made here.**

## Additional ideas

- **Gaia crossmatch check**: do high-dipole-rate objects have a Gaia counterpart?
  Stellar PSF mis-registration is more likely for bright, spatially-resolved stars.
- **CCD position map**: is the high-dipole-rate object always observed on the same
  detector and pixel position, suggesting a bad-column artefact?
- **Dipole fraction per night**: compute the fraction `n_dipoles / n_total` per night
  for the selected objects and compare with the field average from notebook `01c`.
- **Dipole length and angle stability**: for recurring dipoles on the same object,
  is the angle / length stable across visits?  Stable → systematic template offset;
  variable → noise artefact.


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- creation : 2026-05-26
- last update : 2026-05-26

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from astropy.time import Time

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
# ── Input: parquet files from notebook 01c ────────────────────────────────────
DIR_DATA_IN = "data_DIPOLES_01c"

# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "DIPOLES_03"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Input data : {os.path.abspath(DIR_DATA_IN)}")
print(f"Output data: {os.path.abspath(DIR_DATA)}")
print(f"Figures    : {os.path.abspath(DIR_FIGS)}")

# ── DDF definitions (must match 01c) ─────────────────────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

# ── Pre-selection threshold on total visits (all bands combined) ──────────────
# Objects with fewer than MIN_VISITS_TOTAL visits across all bands are excluded
# from the dipole analysis: a ratio 2/2 or 3/3 carries no statistical weight.
# Typical values: 50 (moderate) or 100 (strict).
MIN_VISITS_TOTAL = 20  # ← adjust here (50 or 100 recommended)

# ── Final selection thresholds for "high-dipole" objects ─────────────────────
# Applied AFTER the MIN_VISITS_TOTAL pre-selection.
MIN_DIPOLES = 1  # minimum number of dipole detections among the pre-selected objects
TOP_N_OBJECTS = 10  # max number of objects shown in detailed light curve plots

# ── Plotting style ────────────────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
BAND_ORDER = list("ugrizy")

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save current figure to both PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print(f"Configuration done.  MIN_VISITS_TOTAL={MIN_VISITS_TOTAL}  MIN_DIPOLES={MIN_DIPOLES}")

## 2. Load parquet files from notebook 01c

Reload the per-DDF parquet files.  No API call is made.

In [ ]:
ddf_alerts: dict[str, pd.DataFrame] = {}

for field_name in DEEP_FIELDS:
    pq = os.path.join(DIR_DATA_IN, f"{field_name}_alerts.parquet")
    if not os.path.exists(pq):
        print(f"[{field_name:12s}] parquet not found at {pq} — skipping.")
        ddf_alerts[field_name] = pd.DataFrame()
        continue
    df = pd.read_parquet(pq)

    # --- enforce dtypes --------------------------------------------------
    for col in (
        "r:ra",
        "r:dec",
        "r:midpointMjdTai",
        "r:nDiaSources",
        "r:psfFlux",
        "r:psfFluxErr",
        "r:scienceFlux",
        "r:scienceFluxErr",
        "r:dipoleAngle",
        "r:dipoleLength",
        "r:dipoleChi2",
        "r:dipoleFluxDiff",
    ):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "r:isDipole" in df.columns:
        df["r:isDipole"] = (
            df["r:isDipole"]
            .map(
                lambda v: (
                    True
                    if str(v).strip().lower() in ("true", "1", "yes")
                    else False
                    if str(v).strip().lower() in ("false", "0", "no")
                    else pd.NA
                )
            )
            .astype("boolean")
        )

    ddf_alerts[field_name] = df
    n_tot = len(df)
    n_dip = int(df["r:isDipole"].fillna(False).sum()) if "r:isDipole" in df.columns else 0
    print(f"[{field_name:12s}] {n_tot:6d} alerts  |  {n_dip:5d} dipoles")

# Concatenate all DDFs into a single DataFrame
df_all = pd.concat(
    [d.assign(field=fn) for fn, d in ddf_alerts.items() if not d.empty],
    ignore_index=True,
)
df_all["is_dipole"] = df_all["r:isDipole"].fillna(False).astype(bool)
print(f"\nTotal: {len(df_all):,} alerts across all DDFs")

## 3. Utilities

In [ ]:
def mjd_to_datestr(mjd_array) -> list:
    """
    Convert an array of MJD (TAI) values to ISO date strings 'YYYY-MM-DD'.
    """
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def add_date_axis_on_top(ax, mjd_values: np.ndarray, n_ticks: int = 8) -> None:
    """
    Add a secondary x-axis on top of *ax* showing calendar dates (YYYY-MM-DD).
    """
    finite = mjd_values[np.isfinite(mjd_values)]
    if len(finite) < 2:
        return
    mjd_lo, mjd_hi = float(finite.min()), float(finite.max())
    if mjd_hi <= mjd_lo:
        return
    n_ticks = max(3, min(n_ticks, len(finite)))
    tick_mjd = np.linspace(mjd_lo, mjd_hi, n_ticks)
    tick_lbls = mjd_to_datestr(tick_mjd)

    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(tick_mjd)
    ax_top.set_xticklabels(tick_lbls, rotation=40, ha="left", fontsize=7)
    ax_top.tick_params(axis="x", length=4, pad=2)
    ax_top.set_xlabel("Date (UTC)", fontsize=7, labelpad=6)


print("Utility functions defined.")

## 4. Per-object aggregation and pre-selection on total visits

We first aggregate all alerts by `diaObjectId` to get `n_visits` (total across all bands)
and `n_dipoles`.  We then apply the **pre-selection** `n_visits >= MIN_VISITS_TOTAL` to
discard objects that are too poorly sampled to draw any conclusion about their dipole rate.

The plot below shows the visit count distribution before and after the cut, to verify
that the threshold is appropriate for the current data volume.

In [ ]:
if (
    "r:isDipole" not in df_all.columns
    or "r:diaObjectId" not in df_all.columns
    or "r:diaObjectId" not in df_all.columns
    or "r:nDiaSources" not in df_all.columns
):
    raise RuntimeError("Missing required columns r:isDipole or r:diaObjectId. or r:nDiaSources ")

In [ ]:
# ── Full per-object aggregation (all objects, before any cut) ─────────────────
obj_agg_full = (
    df_all.groupby("r:diaObjectId")
    .agg(
        n_visits=("r:diaSourceId", "count"),
        n_dipoles=("is_dipole", "sum"),
        field=("field", "first"),
        ra=("r:ra", "first"),
        dec=("r:dec", "first"),
        gaia_name=("f:xm_gaiadr3_DR3Name", "first")
        if "f:xm_gaiadr3_DR3Name" in df_all.columns
        else ("r:diaObjectId", "first"),
        simbad_type=("f:xm_simbad_otype", "first")
        if "f:xm_simbad_otype" in df_all.columns
        else ("r:diaObjectId", "first"),
    )
    .reset_index()
)
obj_agg_full["dipole_fraction"] = obj_agg_full["n_dipoles"] / obj_agg_full["n_visits"]

print(f"Total diaObjects (no cut)         : {len(obj_agg_full):,}")
for thr in (5, 10, 20, 50, 100):
    n = (obj_agg_full["n_visits"] >= thr).sum()
    print(f"  n_visits >= {thr:3d}               : {n:,}  ({100 * n / len(obj_agg_full):.1f}%)")

In [ ]:
obj_agg_full

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
df_all["r:nDiaSources"].hist(bins=50, range=(0, 2000), ax=ax, facecolor="b")
ax.set_xlabel("r:nDiaSources")
ax.set_yscale("log")
ax.set_title("number of dia-sources per diaobject ")

In [ ]:
# ── Distribution of n_visits (log-scale) with the cut threshold shown ─────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

nvis = obj_agg_full["n_visits"].values

axes[0].hist(nvis, bins=60, color="steelblue", edgecolor="white", linewidth=0.3)
axes[0].axvline(
    MIN_VISITS_TOTAL, color="tomato", lw=1.5, ls="--", label=f"MIN_VISITS_TOTAL = {MIN_VISITS_TOTAL}"
)
axes[0].set_xlabel("n_visits per diaObject (all bands)")
axes[0].set_ylabel("N diaObjects")
axes[0].set_title("Visit count distribution (linear)")
axes[0].legend(fontsize=8)

bins_log = np.logspace(0, np.log10(nvis.max() + 1), 50)
axes[1].hist(nvis, bins=bins_log, color="steelblue", edgecolor="white", linewidth=0.3)
axes[1].axvline(
    MIN_VISITS_TOTAL, color="tomato", lw=1.5, ls="--", label=f"MIN_VISITS_TOTAL = {MIN_VISITS_TOTAL}"
)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("n_visits per diaObject (all bands)")
axes[1].set_ylabel("N diaObjects")
axes[1].set_title("Visit count distribution (log-log)")
axes[1].legend(fontsize=8)

plt.tight_layout()
savefig(f"nvisits_distribution_threshold{MIN_VISITS_TOTAL}")
plt.show()

In [ ]:
# ── Apply MIN_VISITS_TOTAL pre-selection ──────────────────────────────────────
obj_agg = obj_agg_full[obj_agg_full["n_visits"] >= MIN_VISITS_TOTAL].copy().reset_index(drop=True)

# Restrict df_all to pre-selected objects for all downstream plots
presel_ids = set(obj_agg["r:diaObjectId"])
df_presel = df_all[df_all["r:diaObjectId"].isin(presel_ids)].copy()

print(f"Objects after n_visits >= {MIN_VISITS_TOTAL} cut : {len(obj_agg):,}")
print(f"  with >= 1 dipole  : {(obj_agg['n_dipoles'] >= 1).sum():,}")
print(f"  with >= 3 dipoles : {(obj_agg['n_dipoles'] >= 3).sum():,}")
print(f"  with >= 10 dipoles: {(obj_agg['n_dipoles'] >= 10).sum():,}")
print(f"Alerts in pre-selected objects    : {len(df_presel):,}")
display(obj_agg.sort_values("n_dipoles", ascending=False).head(20))

## 5. Histogram: number of dipoles per object, stacked by band

Each bar = one `diaObjectId` (pre-selected, `n_visits >= MIN_VISITS_TOTAL`).  
The stacked colours show how many dipoles were detected in each band (u/g/r/i/z/y).  
Objects are sorted by total dipole count (descending).

This directly answers: *which well-observed objects concentrate the dipoles, and in which bands?*

In [ ]:
# ── Per-object, per-band dipole count matrix (pre-selected objects only) ──────
df_dip_only = df_presel[df_presel["is_dipole"]].copy()

if "r:band" in df_dip_only.columns and not df_dip_only.empty:
    dip_per_obj_band = (
        df_dip_only.groupby(["r:diaObjectId", "r:band"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=BAND_ORDER, fill_value=0)
    )
    dip_per_obj_band["total"] = dip_per_obj_band.sum(axis=1)
    dip_per_obj_band = dip_per_obj_band.sort_values("total", ascending=False)

    dip_per_obj_band_nonzero = dip_per_obj_band[dip_per_obj_band["total"] > 0].copy()

    print(
        f"Pre-selected objects (n_visits>={MIN_VISITS_TOTAL}) with >=1 dipole: "
        f"{len(dip_per_obj_band_nonzero):,}"
    )
    display(dip_per_obj_band_nonzero.head(20))
else:
    print("Missing r:band column or no dipole alerts in pre-selected objects.")

In [ ]:
# ── Stacked histogram: dipoles per object ─────────────────────────────────────
if "r:band" in df_dip_only.columns and not df_dip_only.empty:
    N_SHOW = 60
    top_df = dip_per_obj_band_nonzero.head(N_SHOW)
    x_pos = np.arange(len(top_df))

    fig, ax = plt.subplots(figsize=(max(12, N_SHOW * 0.22), 5))
    bottom = np.zeros(len(top_df))

    for band in BAND_ORDER:
        if band not in top_df.columns:
            continue
        vals = top_df[band].values.astype(float)
        ax.bar(
            x_pos,
            vals,
            bottom=bottom,
            color=BAND_COLORS[band],
            edgecolor="white",
            linewidth=0.3,
            label=f"band {band}",
            width=0.85,
        )
        bottom += vals

    ax.set_xticks(x_pos)
    ax.set_xticklabels(
        [str(oid)[-6:] for oid in top_df.index],
        rotation=90,
        fontsize=6,
    )
    ax.set_xlabel("diaObjectId (last 6 digits)")
    ax.set_ylabel("Number of dipole detections")
    ax.set_title(
        f"Dipole count per diaObject — top {N_SHOW} objects (stacked by band)\n"
        f"Pre-selection: n_visits >= {MIN_VISITS_TOTAL}  |  "
        f"Objects with >=1 dipole: {len(dip_per_obj_band_nonzero):,}"
    )
    ax.legend(loc="upper right", fontsize=8, ncol=3)
    plt.tight_layout()
    savefig(f"dipole_count_per_object_stacked_band_minvis{MIN_VISITS_TOTAL}")
    plt.show()

In [ ]:
# ── Distribution of dipole count among pre-selected objects ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
counts = dip_per_obj_band_nonzero["total"].values

axes[0].hist(counts, bins=50, color="steelblue", edgecolor="white", linewidth=0.3)
axes[0].set_xlabel("Number of dipole detections per object")
axes[0].set_ylabel("Number of diaObjects")
axes[0].set_title(f"Dipole count distribution (linear)  [n_visits>={MIN_VISITS_TOTAL}]")

axes[1].hist(
    counts,
    bins=np.logspace(0, np.log10(max(counts) + 1), 30),
    color="tomato",
    edgecolor="white",
    linewidth=0.3,
)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("Number of dipole detections per object")
axes[1].set_ylabel("Number of diaObjects")
axes[1].set_title(f"Dipole count distribution (log-log)  [n_visits>={MIN_VISITS_TOTAL}]")

plt.tight_layout()
savefig(f"dipole_count_distribution_minvis{MIN_VISITS_TOTAL}")
plt.show()

## 6. Lorenz-type concentration plot

What fraction of the total dipole budget is carried by the top X% of *well-observed* objects?
Only pre-selected objects (`n_visits >= MIN_VISITS_TOTAL`) are included here.

In [ ]:
sorted_counts = np.sort(dip_per_obj_band_nonzero["total"].values)[::-1]
cum_fraction = np.cumsum(sorted_counts) / sorted_counts.sum()
obj_fraction = np.arange(1, len(sorted_counts) + 1) / len(sorted_counts)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(obj_fraction * 100, cum_fraction * 100, color="steelblue", lw=2)
ax.plot([0, 100], [0, 100], "--", color="grey", lw=1, label="equal distribution")
ax.fill_between(obj_fraction * 100, cum_fraction * 100, obj_fraction * 100, alpha=0.15, color="steelblue")

idx10 = max(1, int(0.10 * len(sorted_counts)))
frac10 = cum_fraction[idx10 - 1] * 100
ax.axvline(10, color="tomato", lw=1, ls=":")
ax.axhline(frac10, color="tomato", lw=1, ls=":")
ax.text(11, frac10 + 1, f"top 10% objects\n→ {frac10:.0f}% of dipoles", color="tomato", fontsize=8)

ax.set_xlabel("Fraction of well-observed objects (%, sorted by dipole count)")
ax.set_ylabel("Cumulative fraction of dipoles (%)")
ax.set_title(f"Dipole concentration — Lorenz curve  [n_visits>={MIN_VISITS_TOTAL}]")
ax.legend(fontsize=8)
plt.tight_layout()
savefig(f"dipole_lorenz_curve_minvis{MIN_VISITS_TOTAL}")
plt.show()

## 7. Select high-dipole-count objects

Among the pre-selected objects (`n_visits >= MIN_VISITS_TOTAL`), we keep those with
`n_dipoles >= MIN_DIPOLES` and sort by `n_dipoles` descending.  The top `TOP_N_OBJECTS`
are inspected in detail in section 8.

In [ ]:
sel = (
    obj_agg[obj_agg["n_dipoles"] >= MIN_DIPOLES]
    .sort_values("n_dipoles", ascending=False)
    .reset_index(drop=True)
)

print(f"Final selection (n_visits>={MIN_VISITS_TOTAL}, n_dipoles>={MIN_DIPOLES}): {len(sel)} objects")
display(
    sel[
        [
            "r:diaObjectId",
            "field",
            "n_visits",
            "n_dipoles",
            "dipole_fraction",
            "ra",
            "dec",
            "gaia_name",
            "simbad_type",
        ]
    ].head(30)
)

sel.to_parquet(os.path.join(DIR_DATA, "selected_high_dipole_objects.parquet"), index=False)
sel.to_csv(os.path.join(DIR_DATA, "selected_high_dipole_objects.csv"), index=False)
print("Selection table saved.")

In [ ]:
# ── 2-D scatter: n_visits vs n_dipoles — pre-selected population ──────────────
fig, ax = plt.subplots(figsize=(7, 5))

for field_name in DEEP_FIELDS:
    sub = obj_agg[obj_agg["field"] == field_name]
    ax.scatter(sub["n_visits"], sub["n_dipoles"], s=12, alpha=0.5, label=field_name)

# Highlight top selected objects
ax.scatter(
    sel["n_visits"].values[:TOP_N_OBJECTS],
    sel["n_dipoles"].values[:TOP_N_OBJECTS],
    s=90,
    marker="*",
    color="gold",
    edgecolors="k",
    linewidths=0.5,
    zorder=5,
    label=f"top {TOP_N_OBJECTS} selected",
)

ax.set_xlabel(f"Total number of visits (n_visits, all >= {MIN_VISITS_TOTAL})")
ax.set_ylabel("Number of dipole detections (n_dipoles)")
ax.set_title(f"Visits vs dipoles per diaObject  [n_visits>={MIN_VISITS_TOTAL}]")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
savefig(f"nvisits_vs_ndipoles_scatter_minvis{MIN_VISITS_TOTAL}")
plt.show()

## 8. Light curve inspection of selected high-dipole objects

For each selected object we produce a figure with **three panels** stacked vertically:

**Top panel** — Light curve (`psfFlux` in nJy):
- All visits: coloured dots by band (BAND_COLORS).
- Dipole visits: same dots surrounded by a grey circle marker.
- Primary x-axis: MJD.  Secondary x-axis (top): date YYYY-MM-DD.

**Middle panel** — Nightly dipole count histogram:
- For each night (floor of MJD), count dipole detections per band (stacked bars).
- Secondary y-axis on the right: cumulative dipole count.

**Bottom panel** — Dipole morphology:
- `dipoleLength` (arcsec) and `dipoleAngle` (degrees) vs time.
- Stable angle → systematic template offset; scattered → noise artefact.

In [ ]:
def plot_object_lightcurve(
    df_obj: pd.DataFrame,
    obj_id: int | str,
    field_name: str,
    n_visits: int,
    n_dipoles: int,
    gaia_name: str = "",
    simbad_type: str = "",
    flux_col: str = "r:psfFlux",
    flux_err_col: str = "r:psfFluxErr",
) -> None:
    """
    Three-panel light curve figure for one diaObject.

    Parameters
    ----------
    df_obj       : DataFrame of all alerts for this diaObject (all bands)
    obj_id       : diaObjectId (for the title)
    field_name   : DDF name
    n_visits     : total number of visits (all bands)
    n_dipoles    : total number of dipole detections
    gaia_name    : Gaia DR3 source name (if available)
    simbad_type  : SIMBAD object type (if available)
    flux_col     : column to plot on the y-axis
    flux_err_col : error column
    """
    df_obj = df_obj.sort_values("r:midpointMjdTai").copy()
    df_obj["is_dipole"] = df_obj["r:isDipole"].fillna(False).astype(bool)
    mjd_all = df_obj["r:midpointMjdTai"].values

    fig, axes = plt.subplots(
        3,
        1,
        figsize=(12, 9),
        gridspec_kw={"height_ratios": [3, 1.5, 1.5]},
        sharex=False,
    )

    # ── Panel 1: light curve ─────────────────────────────────────────────────
    ax1 = axes[0]
    for band in BAND_ORDER:
        sub_b = df_obj[df_obj["r:band"] == band]
        if sub_b.empty:
            continue
        flux = pd.to_numeric(sub_b[flux_col], errors="coerce").values
        ferr = (
            pd.to_numeric(sub_b[flux_err_col], errors="coerce").values
            if flux_err_col in sub_b.columns
            else None
        )
        mjd_b = sub_b["r:midpointMjdTai"].values
        color = BAND_COLORS[band]

        ax1.errorbar(
            mjd_b,
            flux,
            yerr=ferr,
            fmt="o",
            ms=5,
            lw=0.8,
            capsize=2,
            capthick=0.8,
            color=color,
            ecolor=color,
            alpha=0.8,
            label=f"band {band} (n={len(sub_b)})",
        )

        # Dipole visits — grey ring overlay
        sub_dip = sub_b[sub_b["is_dipole"]]
        if not sub_dip.empty:
            flux_dip = pd.to_numeric(sub_dip[flux_col], errors="coerce").values
            ax1.scatter(
                sub_dip["r:midpointMjdTai"].values,
                flux_dip,
                s=120,
                facecolors="none",
                edgecolors="grey",
                linewidths=1.8,
                zorder=5,
                label=(f"dipole {band} (n={len(sub_dip)})" if band == BAND_ORDER[0] else "_nolegend_"),
            )

    ax1.axhline(0, color="k", lw=0.5, ls="--", alpha=0.4)
    ax1.set_ylabel(f"{flux_col.split(':')[1]} (nJy)")
    ax1.legend(loc="best", fontsize=7, ncol=3)

    meta = (
        f"  field={field_name}  n_vis={n_visits}  n_dip={n_dipoles}"
        f"  dip_frac={n_dipoles / max(1, n_visits) * 100:.1f}%"
    )
    if gaia_name and str(gaia_name) not in ("nan", "None", ""):
        meta += f"  Gaia={gaia_name}"
    if simbad_type and str(simbad_type) not in ("nan", "None", ""):
        meta += f"  SIMBAD={simbad_type}"
    ax1.set_title(f"diaObjectId={obj_id}{meta}", fontsize=9)
    add_date_axis_on_top(ax1, mjd_all, n_ticks=8)

    # ── Panel 2: nightly dipole count histogram ───────────────────────────────
    ax2 = axes[1]
    df_dip = df_obj[df_obj["is_dipole"]].copy()

    if not df_dip.empty:
        df_dip["night"] = np.floor(df_dip["r:midpointMjdTai"].values).astype(int)
        night_band = (
            df_dip.groupby(["night", "r:band"])
            .size()
            .unstack(fill_value=0)
            .reindex(columns=BAND_ORDER, fill_value=0)
        )
        night_band["total"] = night_band.sum(axis=1)
        nights_mjd = night_band.index.values.astype(float) + 0.5

        bottom = np.zeros(len(night_band))
        for band in BAND_ORDER:
            if band not in night_band.columns:
                continue
            vals = night_band[band].values.astype(float)
            ax2.bar(
                nights_mjd,
                vals,
                bottom=bottom,
                width=0.8,
                color=BAND_COLORS[band],
                edgecolor="white",
                linewidth=0.3,
                label=f"band {band}",
            )
            bottom += vals

        cum_dip = np.cumsum(night_band["total"].values)
        ax2r = ax2.twinx()
        ax2r.step(
            nights_mjd, cum_dip, where="post", color="k", lw=1.5, ls="--", alpha=0.6, label="cumulative"
        )
        ax2r.set_ylabel("Cumulative dipoles", fontsize=8)
        ax2r.tick_params(axis="y", labelsize=8)

    ax2.set_ylabel("N dipoles per night")
    ax2.set_xlabel("MJD (TAI)")
    ax2.legend(loc="upper left", fontsize=7, ncol=3)

    if len(mjd_all) > 1:
        ax1.set_xlim(mjd_all.min() - 1, mjd_all.max() + 1)
        ax2.set_xlim(ax1.get_xlim())

    # ── Panel 3: dipole morphology ────────────────────────────────────────────
    ax3 = axes[2]
    if not df_dip.empty:
        for band in BAND_ORDER:
            sub_b = df_dip[df_dip["r:band"] == band]
            if sub_b.empty or "r:dipoleLength" not in sub_b.columns:
                continue
            dl = pd.to_numeric(sub_b["r:dipoleLength"], errors="coerce").values
            ax3.scatter(
                sub_b["r:midpointMjdTai"].values,
                dl,
                s=20,
                color=BAND_COLORS[band],
                marker="o",
                label=f"length {band}",
            )

        if "r:dipoleAngle" in df_dip.columns:
            ax3r = ax3.twinx()
            for band in BAND_ORDER:
                sub_b = df_dip[df_dip["r:band"] == band]
                if sub_b.empty:
                    continue
                da = pd.to_numeric(sub_b["r:dipoleAngle"], errors="coerce").values
                ax3r.scatter(
                    sub_b["r:midpointMjdTai"].values,
                    da % 360,
                    s=20,
                    color=BAND_COLORS[band],
                    marker="^",
                    alpha=0.6,
                )
            ax3r.set_ylabel("Dipole angle (deg)", fontsize=8, color="grey")
            ax3r.set_ylim(0, 360)
            ax3r.tick_params(axis="y", labelcolor="grey", labelsize=8)

        ax3.set_ylabel("Dipole length (arcsec)")
        ax3.set_xlabel("MJD (TAI)")
        ax3.legend(loc="upper left", fontsize=7, ncol=3)
        ax3.set_xlim(ax1.get_xlim())

    plt.tight_layout()
    savefig(f"lc_obj_{str(obj_id).replace('/', '_')}")
    plt.show()


print("plot_object_lightcurve() defined.")

In [ ]:
# ── Plot light curves for top selected objects ────────────────────────────────
top_sel = sel.head(TOP_N_OBJECTS)

for _, row in top_sel.iterrows():
    obj_id = row["r:diaObjectId"]
    field = row["field"]
    df_obj = df_all[df_all["r:diaObjectId"] == obj_id].copy()
    if df_obj.empty:
        print(f"No data for object {obj_id} — skipping.")
        continue
    print(
        f"\n=== diaObjectId={obj_id}  field={field}  "
        f"n_visits={row['n_visits']}  n_dipoles={row['n_dipoles']} ==="
    )
    plot_object_lightcurve(
        df_obj=df_obj,
        obj_id=obj_id,
        field_name=field,
        n_visits=int(row["n_visits"]),
        n_dipoles=int(row["n_dipoles"]),
        gaia_name=str(row.get("gaia_name", "")),
        simbad_type=str(row.get("simbad_type", "")),
    )

print("Done.")

## 9. Per-field stacked histograms

Same stacked bar histogram but split per DDF, restricted to pre-selected objects.

In [ ]:
n_fields = len(DEEP_FIELDS)
ncols = min(3, n_fields)
nrows = int(np.ceil(n_fields / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)

for idx, field_name in enumerate(DEEP_FIELDS):
    ax = axes[idx // ncols][idx % ncols]

    # Use only pre-selected objects from this field
    field_ids = set(obj_agg[obj_agg["field"] == field_name]["r:diaObjectId"])
    df_loc = df_presel[(df_presel["field"] == field_name) & (df_presel["is_dipole"])]

    if df_loc.empty or "r:band" not in df_loc.columns:
        ax.set_title(f"{field_name} — no data after pre-selection")
        continue

    dip_f = (
        df_loc.groupby(["r:diaObjectId", "r:band"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=BAND_ORDER, fill_value=0)
    )
    dip_f["total"] = dip_f.sum(axis=1)
    dip_f = dip_f.sort_values("total", ascending=False)

    N_SHOW_F = min(40, len(dip_f))
    top_f = dip_f.head(N_SHOW_F)
    x_pos = np.arange(len(top_f))
    bottom = np.zeros(len(top_f))

    for band in BAND_ORDER:
        if band not in top_f.columns:
            continue
        vals = top_f[band].values.astype(float)
        ax.bar(
            x_pos,
            vals,
            bottom=bottom,
            color=BAND_COLORS[band],
            edgecolor="white",
            linewidth=0.2,
            label=band,
            width=0.85,
        )
        bottom += vals

    ax.set_xticks(x_pos)
    ax.set_xticklabels([str(oid)[-6:] for oid in top_f.index], rotation=90, fontsize=5)
    ax.set_title(
        f"{field_name} — top {N_SHOW_F} objects  ({len(dip_f)} with >=1 dip, n_vis>={MIN_VISITS_TOTAL})",
        fontsize=8,
    )
    ax.set_ylabel("N dipoles")
    ax.legend(loc="upper right", fontsize=6, ncol=3)

for idx in range(n_fields, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.suptitle(
    f"Dipole count per diaObject — per DDF (stacked by band, n_visits>={MIN_VISITS_TOTAL})",
    fontsize=11,
    y=1.01,
)
plt.tight_layout()
savefig(f"dipole_count_per_object_per_ddf_minvis{MIN_VISITS_TOTAL}")
plt.show()

## 10. Gaia crossmatch diagnostic on selected objects

In [ ]:
gaia_cols = ["f:xm_gaiadr3_DR3Name", "f:xm_gaiadr3_PhotGMag", "f:xm_gaiadr3_Plx", "f:xm_gaiadr3_VarFlag"]
avail_gaia = [c for c in gaia_cols if c in df_all.columns]

if avail_gaia:
    gaia_first = df_all.groupby("r:diaObjectId")[avail_gaia].first().reset_index()
    sel_gaia = sel.merge(gaia_first, on="r:diaObjectId", how="left")

    if "f:xm_gaiadr3_PhotGMag" in sel_gaia.columns:
        sel_gaia["f:xm_gaiadr3_PhotGMag"] = pd.to_numeric(sel_gaia["f:xm_gaiadr3_PhotGMag"], errors="coerce")

        fig, ax = plt.subplots(figsize=(6, 4))
        sc = ax.scatter(
            sel_gaia["f:xm_gaiadr3_PhotGMag"],
            sel_gaia["dipole_fraction"] * 100,
            c=np.log10(sel_gaia["n_visits"] + 1),
            cmap="viridis",
            s=40,
            alpha=0.7,
            edgecolors="k",
            linewidths=0.3,
        )
        plt.colorbar(sc, ax=ax, label="log10(n_visits)")
        ax.set_xlabel("Gaia G magnitude")
        ax.set_ylabel("Dipole fraction (%)")
        ax.set_title(f"Dipole fraction vs Gaia G  [n_visits>={MIN_VISITS_TOTAL}]")
        ax.invert_xaxis()
        plt.tight_layout()
        savefig(f"dipole_fraction_vs_gaia_G_minvis{MIN_VISITS_TOTAL}")
        plt.show()
    else:
        print("Gaia G magnitude not available — skipping.")
else:
    print("No Gaia crossmatch columns found in parquet files.")

## 11. CCD position diagnostic for selected objects

In [ ]:
ccd_cols = ["r:detector", "r:x", "r:y"]
have_ccd = all(c in df_all.columns for c in ccd_cols)

if have_ccd:
    top_obj_ids = sel["r:diaObjectId"].head(TOP_N_OBJECTS).tolist()
    df_top = df_all[df_all["r:diaObjectId"].isin(top_obj_ids)].copy()
    df_top["is_dipole"] = df_top["r:isDipole"].fillna(False).astype(bool)
    for col in ("r:x", "r:y"):
        df_top[col] = pd.to_numeric(df_top[col], errors="coerce")

    n_show = min(5, len(top_obj_ids))
    fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 7), squeeze=False)

    for col_idx, obj_id in enumerate(top_obj_ids[:n_show]):
        sub = df_top[df_top["r:diaObjectId"] == obj_id]
        nd = sub[~sub["is_dipole"]]
        dp = sub[sub["is_dipole"]]

        ax = axes[0][col_idx]
        ax.scatter(nd["r:x"], nd["r:y"], s=8, alpha=0.4, color="steelblue", label="non-dipole")
        ax.scatter(
            dp["r:x"],
            dp["r:y"],
            s=30,
            alpha=0.9,
            color="tomato",
            edgecolors="k",
            linewidths=0.5,
            label="dipole",
            zorder=5,
        )
        ax.set_title(f"obj …{str(obj_id)[-6:]}", fontsize=8)
        ax.set_xlabel("x (pix)")
        ax.set_ylabel("y (pix)")
        if col_idx == 0:
            ax.legend(fontsize=7)

        ax2 = axes[1][col_idx]
        all_dets = sorted(sub["r:detector"].dropna().astype(int).unique())
        nd_counts = [((sub["r:detector"].astype(float) == d) & ~sub["is_dipole"]).sum() for d in all_dets]
        dp_counts = [(dp["r:detector"].astype(float) == d).sum() for d in all_dets]
        x_det = np.arange(len(all_dets))
        ax2.bar(x_det - 0.2, nd_counts, 0.4, color="steelblue", label="non-dipole")
        ax2.bar(x_det + 0.2, dp_counts, 0.4, color="tomato", label="dipole")
        ax2.set_xticks(x_det)
        ax2.set_xticklabels(all_dets, rotation=45, fontsize=7)
        ax2.set_xlabel("detector")
        ax2.set_ylabel("N alerts")
        if col_idx == 0:
            ax2.legend(fontsize=7)

    fig.suptitle("CCD position and detector distribution — top selected objects", fontsize=10, y=1.01)
    plt.tight_layout()
    savefig(f"ccd_position_selected_objects_minvis{MIN_VISITS_TOTAL}")
    plt.show()
else:
    print(f"CCD columns not available — skipping.")

## 12. Save summary statistics

In [ ]:
# Full per-object table (pre-selected)
out_path = os.path.join(DIR_DATA, f"all_objects_dipole_stats_minvis{MIN_VISITS_TOTAL}.parquet")
obj_agg.to_parquet(out_path, index=False)
print(f"Saved {len(obj_agg):,} rows → {out_path}")

# Per-object per-band dipole count matrix (pre-selected, >=1 dipole)
dip_per_obj_band_nonzero.to_parquet(
    os.path.join(DIR_DATA, f"dipole_count_per_object_per_band_minvis{MIN_VISITS_TOTAL}.parquet")
)
print("Saved dipole count matrix.")

## 13. Discussion and next steps

### Impact of the `MIN_VISITS_TOTAL` pre-selection

The threshold removes objects seen only once or twice, which would otherwise dominate
the tail of the dipole-count distribution with uninformative 100% dipole fractions.
The log-log distribution of visit counts (section 4) shows the typical power-law shape;
the cut keeps the upper portion of the distribution where statistics are meaningful.

Try `MIN_VISITS_TOTAL = 50` first; tighten to 100 if the selected population is still
noisy, or relax to 20 if the number of surviving objects is too small.

### Key questions addressed

1. **Are dipoles uniformly distributed across well-observed objects?**  The Lorenz curve
   (section 6) gives the answer as a Gini-like scalar.
2. **Which bands dominate?**  Stacked histograms (sections 5 and 9).
3. **Are dipoles concentrated on specific nights?**  Nightly histograms in section 8.
4. **Is the dipole angle/length stable?**  Bottom panel in section 8 light curves.

### Suggested follow-up notebooks

- **`04_dipole_cutouts.ipynb`**: fetch science/template/difference image triplets.
- **`05_dipole_butler.ipynb`**: locate the visits in the Butler and check WCS quality.
- **`06_dipole_seeing.ipynb`**: correlate per-visit dipole rate with seeing and airmass.
